# Anomaly Detection on Well Log Data
## Machine learning for energy applications — Module 5, Exercise: Anomaly Detection

In this exercise we apply unsupervised anomaly detection techniques to well log data. Measurement anomalies in well logs can arise from tool malfunctions, borehole washouts, invasion effects, or genuinely unusual geological formations. Identifying these anomalies is a critical quality-control step before any petrophysical interpretation or machine-learning workflow.

### Learning Objectives
1. Detect unusual measurements in well log data using statistical rules (Z-score, IQR), after recording what data cleaning removed.
2. Apply Isolation Forest, Local Outlier Factor, and One-Class SVM to multivariate well log data, with one score convention: higher = more unusual.
3. Compare the methods and turn their scores into a ranked review list, knowing that agreement between methods is not validation.
4. Visualize flagged intervals on standard well log plots.

**Labs:** Lab 1 covers Parts 1–3. Lab 2 covers Parts 4–6 and ends with a written alert policy.

---
## Overview

Well log data consists of continuous measurements recorded as a function of depth (measured depth, MD). Each curve — gamma ray (GR), bulk density (RHOB), neutron porosity (NPHI), sonic transit time (DT), caliper (CALI), etc. — responds to different rock and fluid properties.

Anomalies in this data can be caused by:
- **Tool malfunctions** — spikes or drop-outs in a single curve.
- **Borehole conditions** — washouts (CALI >> BS) leading to poor pad contact for density or neutron tools.
- **Invasion effects** — drilling fluid infiltrating the formation and altering shallow resistivity readings.
- **Genuine geological anomalies** — thin beds, fractures, or mineralogical changes that produce unusual log responses.

We will explore several anomaly detection methods, from simple univariate statistics to sophisticated machine-learning algorithms, and compare their results.

### How this lab works (read before you start)

- **Protocol: an exploratory review of one fixed table.** Every rule and detector is fitted on the same complete rows that it then flags (scikit-learn's *outlier detection* mode), and the scaler is fitted on those same rows. That is legitimate for reviewing a historical data set; it is not a test of how a detector would perform on a new well. To score new wells you would fit the scaler and the detector on reference wells only (a pipeline does both), choose and freeze the threshold on reference or validation data, and use `LocalOutlierFactor(novelty=True)` for LOF.
- **No labels.** Nothing in this file says which rows are real problems. We can count flags and rank rows for review, but we cannot report precision, recall or accuracy for any method, and agreement between methods is not evidence that they are right.
- **A score is not an alert.** Each detector produces a continuous score; a flag is a separate decision (score above a threshold). In this notebook every score is oriented so that **higher = more unusual**; scikit-learn's `score_samples` uses the opposite sign.
- **`contamination` (and `nu`) is an alert budget that you choose**, not a prevalence the model discovers: `contamination=0.05` flags 5% of the rows because we asked for 5%.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

print("Libraries loaded successfully.")

---
## Part 1: Data Loading and Exploration

We load the well log dataset and perform an initial exploration — understanding the shape, columns, value ranges, and any missing data — and we record exactly what the cleaning step removes.

In [ ]:
# Load the well log data
df = pd.read_csv("well_logs_data.csv")

# YOUR CODE HERE
# Print the shape of the dataset
# Print the column names
# Print the unique well names and count of wells
# Display df.head(10)
# Display df.describe()
# Call df.info()


In [ ]:
# Visualize key well log curves vs depth for the first well
# Hint: Select the first well using df["wellName"].unique()[0]
# Hint: Create a multi-panel plot (1 row, 4 columns) for GR, RHOB, NPHI, DT
# Hint: Use sharey=True so all panels share the same depth axis
# Hint: Invert the y-axis so that depth increases downward

well_name = df["wellName"].unique()[0]
well_df = df[df["wellName"] == well_name].copy()

log_curves = ["GR", "RHOB", "NPHI", "DT"]
colors = ["green", "red", "blue", "purple"]

# YOUR CODE HERE
# Create a figure with 4 subplots (1 row, 4 columns), figsize=(14, 10), sharey=True
# For each curve, plot curve values on x-axis vs MD on y-axis
# Set labels, titles, and invert y-axis


In [ ]:
# Check for missing values and handle them
# Hint: Use df.isnull().sum() to check missing values
# Hint: Use df.dropna(subset=...) to drop rows with NaN in key columns
# Hint: df.groupby("wellName").size() counts the rows per well

key_columns = ["GR", "RHOB", "NPHI", "DT", "CALI", "BS"]

# YOUR CODE HERE
# Print missing values per column
# Drop rows with NaN in key_columns and store in df_clean (use .copy())
# Print how many rows remain after cleaning
# Audit: display a table with the rows per well before and after cleaning,
#        and the share of rows missing each key column in each well
# Print the wells that were lost entirely and the share of rows kept



**What the cleaning step costs.** `dropna` keeps a row only if all six curves are present, and it gives no warning. In this file one of the six curves is missing for two entire wells, so both wells disappear completely and about four fifths of the rows are dropped. Use your audit to find which wells and which curve. Everything below describes only the three remaining wells, not "the field". The lab keeps the six curves so that everyone works on the same table; alternatives are discussed in the debrief. Write the row and well counts before and after cleaning into your notes (Lab 1 deliverable).

---
## Part 2: Statistical Anomaly Detection

We begin with classical statistical approaches:
- **Z-score method** — flags data points that fall more than 3 standard deviations from the mean in any feature.
- **IQR method** — uses the interquartile range to define fences beyond which points are considered outliers.

These methods are simple, interpretable, and computationally inexpensive, and they are the baseline that every learned detector should be compared with. Two cautions from the lecture: the mean and standard deviation are pulled by the outliers themselves (masking; a robust z-score uses the median and MAD instead), and fences computed on a mixture of rock types flag the minority rock type as a whole.

The scaler is fitted on the complete rows: the same table that every rule and detector in this notebook describes (see *How this lab works*).

In [ ]:
# Select key features and standardize
# (the scaler is fitted on the complete rows: the same table every rule and detector below describes)
# Hint: Use StandardScaler().fit_transform() on the selected features

features = ["GR", "RHOB", "NPHI", "DT", "CALI", "BS"]

# YOUR CODE HERE
# Extract feature values: X = df_clean[features].values
# Create a StandardScaler and fit_transform X into X_scaled
# Create a DataFrame df_scaled from X_scaled with columns=features and index=df_clean.index
# Print the shape and display summary statistics of the scaled data



In [ ]:
# Z-score method: flag points where |z| > 3 for any feature
# Hint: Use np.abs(df_scaled) > 3 and .any(axis=1) to create a boolean mask
# Hint: Store the result as a new column in df_clean

z_threshold = 3

# YOUR CODE HERE
# Create z_flag_mask: True where any feature has |z| > z_threshold
# Add column "zscore_flag" to df_clean (1 = flagged, 0 = not flagged)
# Print the number and percentage of flagged rows
# Print the number of flagged rows per feature



In [ ]:
# Visualize z-score flags on well log plots
# Hint: Plot each curve vs depth, then overlay flagged rows as red scatter markers
# Hint: Filter flagged rows using wdf["zscore_flag"] == 1

well_name = df_clean["wellName"].unique()[0]
wdf = df_clean[df_clean["wellName"] == well_name].copy()

# YOUR CODE HERE
# Create a 4-panel well log display (GR, RHOB, NPHI, DT)
# Plot the full log curves as lines
# Overlay the flagged rows as red scatter markers
# Add labels, title, and invert y-axis



In [ ]:
# IQR method: flag points outside [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
# Hint: Use df_clean[features].quantile(0.25) and .quantile(0.75)
# Hint: IQR = Q3 - Q1
# Hint: lower_bound = Q1 - 1.5 * IQR, upper_bound = Q3 + 1.5 * IQR

# YOUR CODE HERE
# Compute Q1, Q3, and IQR for each feature
# Compute lower and upper bounds
# Create outside_fences: a boolean DataFrame, True where a value is outside its feature's bounds
# Create iqr_flag_mask = outside_fences.any(axis=1)
# Add column "iqr_flag" to df_clean (1 = flagged, 0 = not flagged)
# Print the number and percentage of flagged rows
# Print the IQR bounds and the number of flagged rows for each feature



In [ ]:
# Compare Z-score and IQR detections
# Hint: Use boolean logic to find rows flagged by both, only z-score, and only IQR
# Hint: Create a bar chart to visualize the comparison

# YOUR CODE HERE
# Compute: both, only_zscore, only_iqr (from the columns "zscore_flag" and "iqr_flag")
# Print a summary table
# Create a bar chart showing the three categories



In [ ]:
# Review one flagged interval (Lab 1 deliverable)
# Hint: outside_fences["GR"] marks the rows that the IQR rule flags on GR
# Hint: VSH (shale volume) is supplied in the file; compare with df_clean["VSH"].median()
# Hint: to group consecutive flagged rows of one well into depth intervals:
#       run_id = (flag != flag.shift()).cumsum()   # new id whenever the flag switches on or off
#       wdf[flag == 1].groupby(run_id[flag == 1]).agg(top_MD=("MD", "min"), base_MD=("MD", "max"),
#                                                     rows=("MD", "size"), GR=("GR", "median"), ...)

well_name = df_clean["wellName"].unique()[0]
wdf = df_clean[df_clean["wellName"] == well_name]
flag = wdf["iqr_flag"]

# YOUR CODE HERE
# 1) Count the IQR flags from GR: above or below the fences?
#    Compare the median VSH of those rows with the median VSH of all complete rows
# 2) Group the consecutive IQR-flagged rows of this well into depth intervals and display
#    the five thickest (top and base MD, number of rows, median GR, RHOB, NPHI, CALI, VSH)
# 3) Print, per curve, how many rows of the thickest interval are outside the fences



**Review one interval (Lab 1 deliverable).** Find the thickest flagged interval in your printout and look at those depths on the well log display at the start of Part 1 (the same well). Write one sentence: is it more likely a bad measurement or geology, and what evidence would settle it? VSH is a shale-volume estimate supplied in the file; it is derived from the logs, so it can support a judgement but is not an independent label.

Also look back at the scaled statistics and the IQR bounds: which feature has no spread at all, and what does it contribute to the rules and detectors?

---
## Part 3: Isolation Forest

**Isolation Forest** works on a fundamentally different principle: anomalies are *easier to isolate* than normal points. The algorithm builds an ensemble of random trees that recursively split the feature space. Anomalous points tend to be isolated in fewer splits (shorter path lengths).

Key parameters:
- `contamination` — the share of rows to flag: an **alert budget** you choose (for example from the number of intervals a reviewer can check), not the proportion of anomalies the model discovers. It only sets the threshold on the score; it does not change the ranking.
- `n_estimators` — number of isolation trees.
- `random_state` — for reproducibility.

**Score orientation.** In scikit-learn, `score_samples` is *higher for more normal* rows, `decision_function` is `score_samples` minus the threshold `offset_` (set by `contamination`), and `predict` / `fit_predict` return −1 for flagged rows and +1 for the others. The helper below converts every detector in this notebook, once, to **higher = more unusual**, and a check tests that convention against the flags. For Isolation Forest the converted score is the anomaly score *s* of the original paper (near 1: isolated quickly; well below 0.5: ordinary).

In [ ]:
# One score convention for the whole notebook: HIGHER = MORE UNUSUAL.
def unusualness(detector, X=None):
    """Anomaly score of a fitted scikit-learn detector, oriented so that higher = more unusual."""
    if isinstance(detector, LocalOutlierFactor) and not detector.novelty:
        return -detector.negative_outlier_factor_   # outlier mode: scores of the rows it was fitted on
    return -detector.score_samples(X)               # score_samples is higher for more normal rows


def check_orientation(score, flagged):
    """Test the convention: every flagged row must score higher than every unflagged row."""
    score, flagged = np.asarray(score), np.asarray(flagged, dtype=bool)
    assert score[flagged].min() > score[~flagged].max(), "score orientation does not match the flags"
    print(f"Orientation check passed: all {flagged.sum()} flagged rows score higher than the other {(~flagged).sum()}.")

In [ ]:
# Fit Isolation Forest (outlier mode: fitted on the rows it flags)
# Hint: Use IsolationForest(contamination=0.05, n_estimators=100, random_state=42)
# Hint: .fit_predict(X_scaled) returns -1 for flagged rows (outliers) and +1 for the others
# Hint: unusualness(iso_forest, X_scaled) gives the score with higher = more unusual
# Hint: the flag threshold on that scale is -iso_forest.offset_ (rows scoring above it are flagged)

# YOUR CODE HERE
# Create and fit an IsolationForest model on X_scaled; keep the -1/+1 labels as iso_preds
# Print how many rows got each label
# Add column "iso_forest_flag" to df_clean (1 = flagged, 0 = not flagged)
# Add column "iso_forest_score" with unusualness(...) and store the threshold as iso_threshold
# Run check_orientation(df_clean["iso_forest_score"], iso_preds == -1)
# Print the number and percentage of flagged rows, the score range and the threshold



In [ ]:
# Visualize Isolation Forest flags on well log plots
# Hint: Create a 5-panel plot: 4 log curves + 1 score track
# Hint: On the score track, use axvline at x=iso_threshold to show the threshold
# Hint: Use fill_betweenx to shade where the score is above the threshold (higher = more unusual)

well_name = df_clean["wellName"].unique()[0]
wdf = df_clean[df_clean["wellName"] == well_name].copy()

# YOUR CODE HERE
# Create a 5-panel figure (4 log curves + score track)
# Plot each log curve with the flagged rows highlighted
# Plot the Isolation Forest score track with threshold line and shading



In [ ]:
# Sweep the alert budget: contamination = 0.01, 0.05, 0.10
# Hint: Loop over contamination values and fit a new IsolationForest (same random_state) each time
# Hint: For each value store the count, the threshold (-model.offset_) and the set of flagged
#       row labels: set(df_clean.index[preds == -1])
# Hint: set_a <= set_b is True when every element of set_a is also in set_b

contamination_values = [0.01, 0.05, 0.1]

# YOUR CODE HERE
# For each contamination value: fit, count the flagged rows, print count, percentage and threshold
# Check whether each flagged set contains the one before it
# Create a bar chart comparing flagged-row counts across contamination values



**What contamination controls (Lab 1 deliverable).** Write one sentence: what did changing contamination change, and what did it leave unchanged (the scores, the ranking, the flagged sets)? What evidence would you need to choose a value?

---
## Part 4: Local Outlier Factor

**Local Outlier Factor (LOF)** measures the local density deviation of a data point with respect to its neighbours. Points in sparser regions (relative to their neighbours) receive higher LOF scores and are more likely to be flagged as anomalies.

Key parameters:
- `n_neighbors` — the number of neighbours used to compute local density; it decides what counts as "local".
- `contamination` — again an alert budget: LOF flags that share of the rows whatever `n_neighbors` is, so compare *which* rows are flagged, not how many.

LOF is used here in outlier mode: `fit_predict` on the rows being described, with the scores in `negative_outlier_factor_`. To score new wells you would fit `LocalOutlierFactor(novelty=True)` on reference data and call `predict` or `score_samples` on the new rows only. The `unusualness` helper returns the LOF value itself: about 1 for a row as dense as its neighbours, well above 1 for a sparser row.

In [ ]:
# Fit Local Outlier Factor (outlier mode: fit_predict on the rows it flags)
# Hint: Use LocalOutlierFactor(n_neighbors=20, contamination=0.05)
# Hint: .fit_predict(X_scaled) returns -1 for flagged rows and +1 for the others
# Hint: unusualness(lof) returns the LOF value (from negative_outlier_factor_): higher = more unusual
# Hint: the flag threshold on that scale is -lof.offset_

# YOUR CODE HERE
# Create and fit a LocalOutlierFactor model on X_scaled; keep the labels as lof_preds
# Add column "lof_flag" (1 = flagged) and column "lof_score" (unusualness); store lof_threshold
# Run check_orientation(df_clean["lof_score"], lof_preds == -1)
# Print the number and percentage of flagged rows, the score range and the threshold



In [ ]:
# Visualize LOF flags on well log plots
# Hint: Same approach as Isolation Forest but use orange markers for the LOF flags
# Hint: Draw the threshold line at lof_threshold and shade where the score is above it

well_name = df_clean["wellName"].unique()[0]
wdf = df_clean[df_clean["wellName"] == well_name].copy()

# YOUR CODE HERE
# Create a 5-panel figure (4 log curves + LOF score track)
# Plot each log curve with the LOF flags highlighted in orange
# Plot the LOF score track with threshold and shading



In [ ]:
# Sweep n_neighbors with the budget fixed at contamination=0.05: compare flagged sets, not counts
# Hint: For each value store the flagged set: set(df_clean.index[preds == -1])
# Hint: Jaccard overlap = len(a & b) / len(a | b)  (1 = the same rows, 0 = no rows in common)

neighbor_values = [5, 10, 20, 50, 100]

# YOUR CODE HERE
# For each n_neighbors value: fit a LocalOutlierFactor, store the flagged set, print the count
# Build a DataFrame of Jaccard overlaps between every pair of n_neighbors values and display it
# Show it as a heatmap (sns.heatmap with annot=True, vmin=0, vmax=1)



---
## Part 5: One-Class SVM

**One-Class SVM** learns a decision boundary that encompasses the majority of the training data. Points falling outside the boundary are classified as anomalies. It uses a kernel (typically RBF) to map data into a higher-dimensional space where the boundary is easier to construct.

Key parameters:
- `nu` — an upper bound on the share of training rows left outside the boundary (and a lower bound on the share of support vectors). Like `contamination`, it is a budget you choose, not an estimate of how many rows are faulty. The share actually flagged is close to `nu` but not exactly equal to it, because many rows sit almost exactly on the boundary.
- `kernel` — the kernel function (`rbf` is the default and most common choice).
- `gamma` — kernel coefficient: how tightly the boundary wraps the data. `"scale"` (the default) uses 1 / (number of features × variance of X).

Training time grows quickly with the number of rows, so the code fits on a random subsample above 10,000 rows (this table is smaller). The scaler and the SVM sit in one pipeline, so the scaler is always fitted on exactly the rows the SVM is fitted on. One-Class SVM scores have an arbitrary scale (after the sign flip they are negative here); only their order matters, which is why Part 6 compares ranks, not raw scores.

In [ ]:
# Fit One-Class SVM
# Hint: Training time grows quickly with the number of rows. If len(X) > 10000, fit on a random
#       subsample: fit_rows = np.random.default_rng(42).choice(len(X), size=10000, replace=False)
# Hint: make_pipeline(StandardScaler(), OneClassSVM(nu=0.05, kernel="rbf", gamma="scale"))
#       fits the scaler on the same rows as the SVM; fit it on the raw features X[fit_rows]
# Hint: .predict(X) returns -1 for flagged rows and +1 for the others; unusualness(ocsvm, X) gives the score
# Hint: the flag threshold on that scale is -ocsvm[-1].offset_[0]

# YOUR CODE HERE
# Choose the fitting rows (all rows, or a random subsample above 10000 rows)
# Create and fit the pipeline
# Predict and score every row of the table (keep the labels as svm_preds)
# Add columns "ocsvm_flag" and "ocsvm_score" to df_clean; store ocsvm_threshold
# Run check_orientation(df_clean["ocsvm_score"], svm_preds == -1)
# Print the number and percentage of flagged rows, the score range and the threshold



In [ ]:
# Visualize One-Class SVM flags
# Hint: Same approach as the previous methods but use magenta markers
# Hint: Include the score track with the threshold line at ocsvm_threshold

well_name = df_clean["wellName"].unique()[0]
wdf = df_clean[df_clean["wellName"] == well_name].copy()

# YOUR CODE HERE
# Create a 5-panel figure (4 log curves + OCSVM score track)
# Plot each log curve with the OCSVM flags highlighted in magenta
# Plot the score track with threshold and shading



---
## Part 6: Method Comparison

Each anomaly detection method has different strengths:
- **Z-score / IQR** — fast and interpretable but only considers marginal distributions (univariate).
- **Isolation Forest** — efficient, handles multivariate data well, good for high-dimensional datasets.
- **LOF** — captures local density variations, good for datasets with clusters of different densities.
- **One-Class SVM** — effective when the normal data occupies a compact region in feature space.

We now compare how many rows each method flags and the overlap between methods, count votes, and build a **review order** from the scores. There are no labels: agreement between methods tells you which rows to review first, not which rows are wrong.

In [ ]:
# Compare all methods — count of flagged rows and overlap
# Hint: Define lists of method column names and display names
# Hint: Build a pairwise overlap matrix using nested loops, starting from
#       pd.DataFrame(0, index=method_names, columns=method_names)
# Hint: Visualize the overlap with a heatmap (sns.heatmap)

methods = ["zscore_flag", "iqr_flag", "iso_forest_flag", "lof_flag", "ocsvm_flag"]
method_names = ["Z-Score", "IQR", "Isolation Forest", "LOF", "One-Class SVM"]

# YOUR CODE HERE
# Print the number of flagged rows for each method
# Create a pairwise overlap matrix (DataFrame): rows flagged by both methods
# Display the overlap matrix
# Create a heatmap visualization of the overlap



In [ ]:
# Summary visualization: which depths are flagged by each method
# Hint: Create a multi-panel plot with GR as reference + one binary flag track per method
# Hint: Use fill_betweenx to show the flags as coloured depth strips

well_name = df_clean["wellName"].unique()[0]
wdf = df_clean[df_clean["wellName"] == well_name].copy()

# YOUR CODE HERE
# Create a figure with 6 panels: GR reference + 5 method flag tracks
# First panel: plot GR vs depth
# Remaining panels: for each method, use fill_betweenx to show its flags



In [ ]:
# Count votes: how many of the five methods flag each row
# Hint: Sum the flag columns to get a vote count per row
# Hint: Mark rows with votes >= 2 (a review order, not a verdict)
# Hint: Visualize on well logs with colour-coded scatter by vote count

# YOUR CODE HERE
# Compute df_clean["votes"] = sum of the five flag columns
# Create column "consensus_flag" (votes >= 2)
# Print the distribution of votes (how many rows have 0, 1, 2, ... 5 votes)
# Print the number of rows flagged by 2 or more methods
#
# Visualize: 5-panel plot (4 log curves + vote count track)
# On the log curves, scatter the rows with votes >= 2 coloured by vote count (use cmap="Reds")
# On the vote track, use fill_betweenx and draw a line at 2 votes



In [ ]:
# A review order from the three learned detectors that does not depend on their thresholds
# Hint: every score column is already oriented higher = more unusual
# Hint: df_clean[score_cols].rank(pct=True) converts each column to 0-1 percentile ranks
#       (the same as scipy.stats.rankdata(v) / len(v) on the lecture slide)
# Hint: average the three percentile ranks per row, then sort from high to low

score_cols = ["iso_forest_score", "lof_score", "ocsvm_score"]
review_capacity = 50   # e.g. the number of rows a petrophysicist can review in a day

# YOUR CODE HERE
# Compute the percentile ranks and their mean per row; store it as df_clean["review_rank"]
# Take the top review_capacity rows as review_list
# Print how many of them come from each well, and their vote counts
# Display the first 10 rows (well, MD, GR, RHOB, NPHI, DT, CALI, votes, review_rank)



### What this comparison can and cannot tell you

- **No precision, recall or accuracy.** The file has no labels, so none of these can be computed for any method. Overlap and votes measure agreement; methods that share assumptions (the z-score and IQR rules) agree for the same reasons and can be wrong together.
- **Stability is not correctness.** The Jaccard overlaps in Part 4 show whether a ranking is reproducible, not whether it is right.
- **Evidence comes from review.** Freeze the detector, split its ranking into bands, and let a petrophysicist review a fixed number of rows from *every* band, unflagged rows included, without seeing the scores. Those labels estimate the share of real problems in each band and how many lie below the threshold.
- **Your alert policy (Lab 2 deliverable, half a page):** what is scored (rows, features, wells); the score and its direction; the threshold and why (review capacity, cost or reviewed intervals, not "contamination = 5%"); rules such as merging adjacent flagged rows into one interval and a minimum interval thickness; who reviews, within what time; what evidence exists and what does not (no labels yet); when the policy is reviewed.

---
## Summary and Key Takeaways

In this exercise we applied five anomaly detection methods to well log data:

1. **Z-score** — simple univariate thresholding; fast but misses multivariate anomalies, and the mean and SD are pulled by the outliers themselves.
2. **IQR** — quartile-based fences, less affected by extreme values; still univariate, and on a mixture of rock types it flags the minority rock type (here, shale-rich intervals).
3. **Isolation Forest** — efficient tree-based method that handles multivariate patterns well.
4. **Local Outlier Factor** — density-based method sensitive to local neighbourhood structure and to `n_neighbors`.
5. **One-Class SVM** — boundary-based method; works well when normal data is compact.

**Key observations:**
- Record what cleaning removed before interpreting any flag: here two of the five wells were dropped, so every result describes three wells.
- A score is not an alert. Every score here is oriented higher = more unusual; the flag is a separate threshold decision.
- `contamination` and `nu` are alert budgets we chose: they fix how many rows are flagged, not how many are bad. With the budget fixed, LOF's `n_neighbors` changed *which* rows were flagged.
- No single method is universally best. Each method has different assumptions about what constitutes an anomaly, and agreement between methods gives a review order, not validation.
- Without labels we cannot report precision or recall. A defensible threshold comes from review capacity or reviewed intervals; for new wells it is chosen on reference or validation data and frozen before use.
- Domain knowledge is essential: a caliper washout (CALI >> BS) is a known cause of unreliable density/neutron readings, and this physics-based check should complement statistical methods.

**Ideas for further exploration:**
- Add a physics-based check: flag depths where CALI significantly exceeds BS (borehole washout).
- Keep all five wells by dropping DT (and the constant BS), or analyse each well separately, and compare the flags.
- Novelty mode: fit the scaler and a detector on two wells, set the threshold from a high quantile of their scores, then score the third well.
- Use DBSCAN for density-based clustering and anomaly detection.
- Apply autoencoders (neural network) for reconstruction-error-based anomaly detection.
- Evaluate the ranking against reviewed intervals: a blind review of flagged and unflagged rows, sampled from every score band.